# 46-1 Named Entity Recognition (NER) con PySpark y spaCy

Este notebook demuestra cómo extraer entidades nombradas (personas, organizaciones, lugares, etc.) de textos en inglés usando spaCy a través de UDFs de PySpark, sin dependencias de Spark NLP (John Snow Labs).

## Objetivo
Identificar y clasificar entidades nombradas en un conjunto de textos usando el modelo `en_core_web_sm` de spaCy.

## Flujo del notebook
1. Instalación de dependencias (spaCy + modelo de inglés).
2. Importación de módulos PySpark y spaCy.
3. Creación de la sesión Spark.
4. Creación de datos de ejemplo (textos con entidades).
5. Extracción de entidades con spaCy via UDF.
6. Análisis y conteo de entidades por tipo.

## Antes de ejecutar
- Verifica que la sesión de Spark esté activa.
- Ejecuta las celdas en orden para mantener dependencias.


## 1. Instalación de dependencias
Se instala spaCy y el modelo de inglés `en_core_web_sm` para NER. No se requiere Spark NLP.


In [ ]:
%pip install numpy pandas matplotlib seaborn spacy
!python -m spacy download en_core_web_sm
%restart_python

## 2. Importación de módulos
Se importan PySpark (sesión, tipos, funciones) y spaCy para el reconocimiento de entidades nombradas.


In [ ]:
import findspark
findspark.init('/Users/joseaguilar/Documents/Development/spark/spark-3.5.1-bin-hadoop3')
from pyspark import *
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, ArrayType
from pyspark.sql.functions import col, udf, explode, lit
from pyspark.ml import Pipeline
import spacy
import os

## 3. Creación de la sesión Spark
Se inicializa una `SparkSession` básica sin dependencias externas de NLP.


In [ ]:
spark = SparkSession.builder \
    .appName("ner_spacy") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
import sys
print(f"Python version: {sys.version}")

## 4. Carga del modelo spaCy y definición de UDFs
Se carga el modelo `en_core_web_sm` de spaCy y se definen UDFs para extraer entidades nombradas de cada texto. La UDF retorna un array de structs con el texto de la entidad y su tipo (PERSON, ORG, GPE, etc.).


In [ ]:
# Cargar modelo spaCy
nlp = spacy.load("en_core_web_sm")

# Esquema de salida para las entidades: array de structs (texto, tipo)
entity_schema = ArrayType(StructType([
    StructField("entity", StringType(), False),
    StructField("label", StringType(), False)
]))

# UDF que extrae entidades nombradas de un texto
@udf(entity_schema)
def extract_entities(text):
    if text is None:
        return []
    doc = nlp(text)
    return [(ent.text, ent.label_) for ent in doc.ents]

print("Modelo spaCy cargado correctamente")
print(f"Tipos de entidades soportados: {nlp.get_pipe('ner').labels}")

## 5. Datos de ejemplo
Se crean textos en inglés que contienen entidades nombradas de distintos tipos: personas (PERSON), organizaciones (ORG), lugares (GPE), etc.


In [ ]:

# Crear un conjunto de textos para analizar
texts = [
    ("Apple Inc. is looking to buy a startup in the United States.",),
    ("Barack Obama was the 44th President of the United States.",),
    ("Elon Musk founded SpaceX, an aerospace manufacturer and space transportation company.",),
    ("The Amazon rainforest is the largest tropical rainforest in the world.",),
    ("Google was founded by Larry Page and Sergey Brin while they were Ph.D. students at Stanford University.",)
]

# Crear un DataFrame
df_ner = spark.createDataFrame(texts, ["texto"])

## 6. Extracción de entidades nombradas con spaCy
Se aplica la UDF de spaCy a cada texto para extraer las entidades, luego se hace `explode` para obtener una fila por entidad. Se muestran el texto original, la entidad detectada y su tipo.


In [ ]:
# Aplicar la UDF de NER al DataFrame
df_with_entities = df_ner.withColumn("entidades", extract_entities(col("texto")))

# Explode para ver una entidad por fila
df_entities_flat = df_with_entities.select(
    "texto",
    explode("entidades").alias("entidad")
).select(
    "texto",
    col("entidad.entity").alias("entidad_texto"),
    col("entidad.label").alias("tipo_entidad")
)

df_entities_flat.show(truncate=False)

In [ ]:
## 7. Análisis de entidades por tipo
Se agrupan las entidades detectadas por tipo (PERSON, ORG, GPE, etc.) para contar cuántas hay de cada categoría.


In [ ]:
# Conteo de entidades por tipo
print("=== Entidades por tipo ===")
df_entities_flat.groupBy("tipo_entidad").count().orderBy(col("count").desc()).show()

# Entidades únicas por tipo
print("=== Entidades únicas por tipo ===")
df_entities_flat.groupBy("tipo_entidad", "entidad_texto").count() \
    .orderBy("tipo_entidad", col("count").desc()).show(truncate=False)